In [30]:
import math
import torch
import torch.nn as nn
from modules import *
from einops import einsum, rearrange

In [76]:
class TransformerLM(nn.Module):
    def __init__(
            self, 
            vocab_size: int,
            context_length: int,
            d_model: int,
            num_layers: int,
            num_heads: int,
            d_ff: int,
            rope_theta: float,
            device=None,
            dtype=None
        ):
        super(TransformerLM, self).__init__()

        self.vocab_size = vocab_size
        self.context_length = context_length
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.rope_theta = rope_theta

        self.token_embedding = EmbeddingLayer(num_embeddings=self.vocab_size, embedding_dim=self.d_model, device=device, dtype=dtype)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=self.d_model,
                num_heads=self.num_heads,
                d_ff=self.d_ff,
                max_seq_len=self.context_length,
                theta=self.rope_theta,
                device=device,
                dtype=dtype
            ) for _ in range(self.num_layers)
        ])
        self.norm = RMSNormLayer(d_model=self.d_model, device=device, dtype=dtype)
        self.linear = LinearLayer(in_features=self.d_model, out_features=self.vocab_size, device=device, dtype=dtype)

    def forward(x=None,token_ids=None):
        x = self.token_embedding(token_ids)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = self.linear(x)
        return x

In [77]:
# vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta, device, dtype
tf_lm = TransformerLM(10, 5, 3, 3, 1, 3, 1000)
# for i in range(3):
#     print(f'{i}: RMS Norm 1 (g)', tf_lm.blocks[i].ffn.W1.W.shape)
#     print(f'{i}: RMS Norm 2 (g)', tf_lm.blocks[i].multihead.Query.W.shape)
tf_lm.norm.g.shape

torch.Size([3])

In [ ]:
    new_state_dict = {}
    for i in range(5):
        prefix_theirs = f"layers.{i}."
        prefix_mine = f"blocks.{i}."

        new_state_dict[prefix_mine + "rmsnorm1.g"] = weights[prefix_theirs + "ln1.weight"]
        new_state_dict[prefix_mine + "rmsnorm2.g"] = weights[prefix_theirs + "ln2.weight"]
        new_state_dict[prefix_mine + "multihead.Query.W"] = weights[prefix_theirs + "attn.q_proj.weight"]
        new_state_dict[prefix_mine + "multihead.Key.W"] = weights[prefix_theirs + "attn.k_proj.weight"]
        new_state_dict[prefix_mine + "multihead.Value.W"] = weights[prefix_theirs + "attn.v_proj.weight"]
        new_state_dict[prefix_mine + "multihead.Output.W"] = weights[prefix_theirs + "attn.output_proj.weight"]
        new_state_dict[prefix_mine + "ffn.W1.W"] = weights[prefix_theirs + "ffn.w1.weight"]
        new_state_dict[prefix_mine + "ffn.W2.W"] = weights[prefix_theirs + "ffn.w2.weight"]
        new_state_dict[prefix_mine + "ffn.W3.W"] = weights[prefix_theirs + "ffn.w3.weight"]

    new_state_dict["embedding.E"] = weights["token_embeddings.weight"]
    new_state_dict["norm.g"] = weights["ln_final.weight"]
    new_state_dict["lm_head.W"] = weights["lm_head.weight"]

In [ ]:
new_state_dict = {}
for i in range(num_):
    prefix_theirs = f"layers.{i}."
    prefix_mine = f"blocks.{i}."

    new_state_dict[prefix_mine + "rmsnorm1.g"] = weights[prefix_theirs + "ln1.weight"]
    new_state_dict[prefix_mine + "rmsnorm2.g"] = weights[prefix_theirs + "ln2.weight"]
    new_state_dict[prefix_mine + "multihead.Query.W"] = weights[prefix_theirs + "attn.q_proj.weight"]
    new_state_dict[prefix_mine + "multihead.Key.W"] = weights[prefix_theirs + "attn.k_proj.weight"]
    new_state_dict[prefix_mine + "multihead.Value.W"] = weights[prefix_theirs + "attn.v_proj.weight"]
    new_state_dict[prefix_mine + "multihead.Output.W"] = weights[prefix_theirs + "attn.output_proj.weight"]
    new_state_dict[prefix_mine + "ffn.W1.W"] = weights[prefix_theirs + "ffn.w1.weight"]
    new_state_dict[prefix_mine + "ffn.W2.W"] = weights[prefix_theirs + "ffn.w2.weight"]
    new_state_dict[prefix_mine + "ffn.W3.W"] = weights[prefix_theirs + "ffn.w3.weight"]

new_state_dict["token_embedding.E"] = weights["token_embeddings.weight"]
new_state_dict["norm.g"] = weights["ln_final.weight"]
new_state_dict["linear.W"] = weights["lm_head.weight"]

In [78]:
new_state_dict = {}
for i in range(3):
    prefix_theirs = f"layers.{i}."
    prefix_mine = f"blocks.{i}."

    new_state_dict[prefix_mine + "rmsnorm1.g"] = torch.randn(size=(3,))
    new_state_dict[prefix_mine + "rmsnorm2.g"] = torch.randn(size=(3,))
    new_state_dict[prefix_mine + "multihead.Query.W"] = torch.randn(size=(3,3))
    new_state_dict[prefix_mine + "multihead.Key.W"] = torch.randn(size=(3,3))
    new_state_dict[prefix_mine + "multihead.Value.W"] = torch.randn(size=(3,3))
    new_state_dict[prefix_mine + "multihead.Output.W"] = torch.randn(size=(3,3))
    new_state_dict[prefix_mine + "ffn.W1.W"] = torch.randn(size=(3,3))
    new_state_dict[prefix_mine + "ffn.W2.W"] = torch.randn(size=(3,3))
    new_state_dict[prefix_mine + "ffn.W3.W"] = torch.randn(size=(3,3))

new_state_dict["token_embedding.E"] = torch.randn(size=(10,3))
new_state_dict["norm.g"] = torch.randn(size=(3,))
new_state_dict["linear.W"] = torch.randn(size=(10,3))

In [79]:
tf_lm.load_state_dict(new_state_dict)

<All keys matched successfully>